# EDA on Online Retail Sales

**OASIS INFOBYTE SIP — Data Analytics Level 1 • Task 1**

This notebook follows a reproducible **inspect → clean → analyse → visualise → interpret → recommend** workflow. Each business visualisation is immediately followed by its observation and business implication so the reader does not have to search elsewhere for the interpretation.

### Dataset limitations
- **Age/gender analysis:** impossible because the supplied dataset has no age or gender fields. No values are invented.
- **Product-category analysis:** impossible because the supplied dataset has no dedicated product-category field. Categories are not subjectively inferred.
- **Best-selling product:** defined as the product with the highest total **Quantity Sold**. Revenue ranking is analysed separately.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, Markdown

sns.set_theme(style='whitegrid')

# Make paths reliable whether the notebook is launched from the project root or notebooks/.
BASE = Path.cwd()
if not (BASE / 'data' / 'raw' / 'online_retail.csv').exists():
    if (BASE.parent / 'data' / 'raw' / 'online_retail.csv').exists():
        BASE = BASE.parent
    else:
        raise FileNotFoundError(
            'Could not locate data/raw/online_retail.csv. Run the notebook from '
            'DataAnalytics-L1-EDARetailSales or its notebooks folder.'
        )

RAW_DATA = BASE / 'data' / 'raw' / 'online_retail.csv'
CLEANED_DATA = BASE / 'data' / 'cleaned' / 'online_retail_cleaned.csv'
OUT = BASE / 'outputs'
OUT.mkdir(parents=True, exist_ok=True)
CLEANED_DATA.parent.mkdir(parents=True, exist_ok=True)

df_raw = pd.read_csv(RAW_DATA, encoding='latin1')
df_raw.columns = df_raw.columns.str.strip()

for col in ['Description', 'Country', 'StockCode', 'InvoiceNo']:
    df_raw[col] = df_raw[col].astype('string').str.strip()

df_raw['InvoiceDate'] = pd.to_datetime(df_raw['InvoiceDate'], errors='coerce')

print(f'Raw shape: {df_raw.shape}')
print('\nColumn data types:')
print(df_raw.dtypes.to_string())
print(f'\nExact duplicate rows: {df_raw.duplicated().sum()}')
print('\nMissing values:')
print(df_raw.isna().sum().sort_values(ascending=False).to_string())

## 1. Data cleaning

The analysis removes exact duplicates, cancellation invoices, non-positive quantities/prices and invalid dates. Revenue is calculated as `Quantity × UnitPrice`. Missing `CustomerID` values are retained because customer-level analysis is not required for the core task.

In [ ]:
df = df_raw.drop_duplicates().copy()
df['is_cancelled'] = df['InvoiceNo'].astype('string').str.upper().str.startswith('C', na=False)
df['Revenue'] = df['Quantity'] * df['UnitPrice']

sales_df = df[
    (~df['is_cancelled'])
    & (df['Quantity'] > 0)
    & (df['UnitPrice'] > 0)
    & df['InvoiceDate'].notna()
].copy()

sales_df.to_csv(CLEANED_DATA, index=False)

print(f'Clean sales rows: {len(sales_df):,}')
print(f'Total revenue: £{sales_df['Revenue'].sum():,.2f}')
print(f'Unique invoices: {sales_df['InvoiceNo'].nunique():,}')
print(f'Unique products (stock codes): {sales_df['StockCode'].nunique():,}')
print(f'Unique countries: {sales_df['Country'].nunique():,}')

## 2. Descriptive statistics

The OASIS requirement asks for mean, median, mode and standard deviation for numerical columns. The business-relevant numerical variables are **Quantity, UnitPrice and Revenue**; the technical `index` field is excluded.

In [ ]:
numeric_cols = ['Quantity', 'UnitPrice', 'Revenue']
stats = pd.DataFrame({
    'Mean': sales_df[numeric_cols].mean(),
    'Median': sales_df[numeric_cols].median(),
    'Mode': sales_df[numeric_cols].mode().iloc[0],
    'Standard Deviation': sales_df[numeric_cols].std()
}).round(2)

display(stats)

display(Markdown(
    '**Interpretation:** The means are materially above the medians for Quantity, UnitPrice and Revenue, '
    'while the standard deviations are large relative to the medians. This indicates right-skewed distributions '
    'and substantial transaction-level variability, so the median should be considered alongside the mean when '
    'describing a typical transaction.'
))

## 3. Monthly and quarterly sales trends

In [ ]:
sales_df['YearMonth'] = sales_df['InvoiceDate'].dt.to_period('M')
monthly_revenue = sales_df.groupby('YearMonth')['Revenue'].sum()
quarterly_revenue = sales_df.groupby(sales_df['InvoiceDate'].dt.to_period('Q'))['Revenue'].sum()

fig, ax = plt.subplots(figsize=(12, 5))
monthly_revenue.plot(ax=ax, marker='o')
ax.set_title('Monthly Revenue Trend')
ax.set_xlabel('Month')
ax.set_ylabel('Revenue (£)')
ax.tick_params(axis='x', rotation=45)
fig.tight_layout()
fig.savefig(OUT / 'monthly_revenue_trend.png', dpi=180, bbox_inches='tight')
display(fig)
plt.close(fig)

peak_month = monthly_revenue.idxmax()
peak_month_value = monthly_revenue.max()
display(Markdown(
    f'**📌 Observation:** **{peak_month}** generated the highest monthly revenue at '
    f'**£{peak_month_value:,.2f}**. The pattern shows a strong late-year sales peak.\n\n'
    '**💼 Business implication:** Inventory, fulfilment capacity and promotional planning should be '
    'strengthened ahead of the strongest seasonal period.'
))

fig, ax = plt.subplots(figsize=(10, 5))
quarterly_revenue.plot(kind='bar', ax=ax)
ax.set_title('Quarterly Revenue Trend')
ax.set_xlabel('Quarter')
ax.set_ylabel('Revenue (£)')
ax.tick_params(axis='x', rotation=0)
fig.tight_layout()
fig.savefig(OUT / 'quarterly_revenue_trend.png', dpi=180, bbox_inches='tight')
display(fig)
plt.close(fig)

peak_quarter = quarterly_revenue.idxmax()
peak_quarter_value = quarterly_revenue.max()
display(Markdown(
    f'**📌 Observation:** **{peak_quarter}** was the strongest quarter, generating '
    f'**£{peak_quarter_value:,.2f}**.\n\n'
    '**💼 Business implication:** Quarterly planning should account for concentrated demand in the final quarter.'
))

## 4. Monthly Average Order Value (AOV)

In [ ]:
invoice_revenue = sales_df.groupby(['YearMonth', 'InvoiceNo'])['Revenue'].sum()
monthly_aov = invoice_revenue.groupby(level=0).mean()

fig, ax = plt.subplots(figsize=(12, 5))
monthly_aov.plot(ax=ax, marker='o')
ax.set_title('Monthly Average Order Value (AOV)')
ax.set_xlabel('Month')
ax.set_ylabel('Average Order Value (£)')
ax.tick_params(axis='x', rotation=45)
fig.tight_layout()
fig.savefig(OUT / 'monthly_aov.png', dpi=180, bbox_inches='tight')
fig.savefig(OUT / 'monthly_aov.svg', bbox_inches='tight')
display(fig)
plt.close(fig)

high_aov_month = monthly_aov.idxmax()
low_aov_month = monthly_aov.idxmin()
display(Markdown(
    f'**📌 Observation:** Monthly AOV ranged from **£{monthly_aov.min():,.2f}** in **{low_aov_month}** '
    f'to **£{monthly_aov.max():,.2f}** in **{high_aov_month}**.\n\n'
    '**💼 Business implication:** AOV should be monitored separately from total revenue because a month can '
    'generate strong revenue through more orders without having the highest value per order.'
))

## 5. Country analysis

In [ ]:
country_revenue = sales_df.groupby('Country')['Revenue'].sum().sort_values(ascending=False)
country_orders = sales_df.groupby('Country')['InvoiceNo'].nunique().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(10, 6))
country_revenue.head(10).sort_values().plot(kind='barh', ax=ax)
ax.set_title('Top 10 Countries by Revenue')
ax.set_xlabel('Revenue (£)')
ax.set_ylabel('Country')
fig.tight_layout()
fig.savefig(OUT / 'top_10_countries_by_revenue.png', dpi=180, bbox_inches='tight')
display(fig)
plt.close(fig)

uk_share = country_revenue.get('United Kingdom', 0) / country_revenue.sum() * 100
display(Markdown(
    f'**📌 Observation:** The **United Kingdom** contributed approximately **{uk_share:.1f}%** of total revenue, '
    'showing that revenue is highly concentrated in the home market.\n\n'
    '**💼 Business implication:** The UK remains the core market, while international markets should be evaluated '
    'for targeted expansion rather than treated as equally important.'
))

fig, ax = plt.subplots(figsize=(10, 6))
country_orders.head(10).sort_values().plot(kind='barh', ax=ax)
ax.set_title('Top 10 Countries by Orders')
ax.set_xlabel('Unique Invoices')
ax.set_ylabel('Country')
fig.tight_layout()
fig.savefig(OUT / 'top_10_countries_by_orders.png', dpi=180, bbox_inches='tight')
display(fig)
plt.close(fig)

display(Markdown(
    f'**📌 Observation:** The **United Kingdom** also recorded the largest number of unique invoices '
    f'(**{country_orders.iloc[0]:,}**), reinforcing its role as the dominant market.\n\n'
    '**💼 Business implication:** Operational capacity and customer-retention initiatives should prioritise the UK.'
))

## 6. Product analysis

Non-product/service lines such as postage, carriage, discounts, fees and adjustments are excluded from product rankings. This prevents `DOTCOM POSTAGE` and similar non-product lines from appearing as products.

In [ ]:
non_product_pattern = r'(POSTAGE|CARRIAGE|COURIER|DISCOUNT|MANUAL|FEE|ADJUSTMENT|SAMPLES?)'
product_df = sales_df[
    ~sales_df['Description'].fillna('').str.upper().str.contains(
        non_product_pattern, regex=True, na=False
    )
].copy()

product_units = product_df.groupby('Description')['Quantity'].sum().sort_values(ascending=False)
product_revenue = product_df.groupby('Description')['Revenue'].sum().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(10, 6))
product_units.head(10).sort_values().plot(kind='barh', ax=ax)
ax.set_title('Top 10 Products by Quantity Sold')
ax.set_xlabel('Units Sold')
ax.set_ylabel('Product')
fig.tight_layout()
fig.savefig(OUT / 'top_10_products_by_units.png', dpi=180, bbox_inches='tight')
display(fig)
plt.close(fig)

best_units_product = product_units.index[0]
best_units_value = product_units.iloc[0]
display(Markdown(
    f'**📌 Observation:** **{best_units_product}** was the best-selling product by quantity, with '
    f'**{best_units_value:,} units sold**.\n\n'
    '**💼 Business implication:** High-volume products should receive close inventory monitoring because '
    'stock-outs can affect a large number of transactions.'
))

fig, ax = plt.subplots(figsize=(10, 6))
product_revenue.head(10).sort_values().plot(kind='barh', ax=ax)
ax.set_title('Top 10 Products by Revenue')
ax.set_xlabel('Revenue (£)')
ax.set_ylabel('Product')
fig.tight_layout()
fig.savefig(OUT / 'top_10_products_by_revenue.png', dpi=180, bbox_inches='tight')
display(fig)
plt.close(fig)

best_revenue_product = product_revenue.index[0]
best_revenue_value = product_revenue.iloc[0]
display(Markdown(
    f'**📌 Observation:** **{best_revenue_product}** generated the highest product revenue at '
    f'**£{best_revenue_value:,.2f}** after excluding non-product/service lines.\n\n'
    '**💼 Business implication:** Revenue leaders should be assessed separately from volume leaders because '
    'high unit sales do not necessarily produce the highest revenue.'
))

## 7. Correlation analysis

In [ ]:
corr = sales_df[['Quantity', 'UnitPrice', 'Revenue']].corr()

fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='Blues', ax=ax)
ax.set_title('Correlation Heatmap')
fig.tight_layout()
fig.savefig(OUT / 'correlation_heatmap.png', dpi=180, bbox_inches='tight')
display(fig)
plt.close(fig)

qr = corr.loc['Quantity', 'Revenue']
display(Markdown(
    f'**📌 Observation:** Quantity and Revenue show a strong positive correlation (**r = {qr:.2f}**). '
    'This is expected because Revenue is mechanically calculated as `Quantity × UnitPrice`; therefore, '
    'the correlation should not be interpreted as an independent causal relationship.\n\n'
    '**💼 Business implication:** The heatmap is useful as a consistency check, but business decisions should '
    'combine correlation with the underlying pricing and volume measures.'
))

## 8. Overall findings and recommendations

### Key findings
- Total cleaned sales revenue is approximately **£10.67 million** across **19,960 unique invoices**.
- Revenue is concentrated in the **United Kingdom**, which contributes approximately **84.6%** of total revenue.
- **November 2011** is the peak month by revenue, at approximately **£1.51 million**.
- **2011 Q4** is the strongest quarter, at approximately **£3.30 million**.
- Monthly AOV varies substantially, so revenue and order value should be monitored as separate KPIs.
- The best-selling product by quantity is **PAPER CRAFT , LITTLE BIRDIE**, while **REGENCY CAKESTAND 3 TIER** is the top revenue-generating product after excluding non-product lines.
- The strong Quantity–Revenue correlation is partly mechanical because Revenue is defined as Quantity × UnitPrice.

### Dataset limitations
Age/gender analysis and dedicated product-category analysis are explicitly documented as impossible from the supplied fields. No demographic or category values are invented.

### Recommendations
1. Prepare inventory and fulfilment capacity ahead of late-year demand.
2. Protect the UK core market while testing targeted international growth.
3. Monitor AOV alongside revenue to distinguish higher order volume from higher basket value.
4. Prioritise stock availability for high-volume products.
5. Use revenue and quantity rankings together when making product decisions.
6. Treat correlation findings carefully where one variable is mathematically derived from another.

In [ ]:
# Final output validation: confirms that every required visualisation was written to outputs/.
required_outputs = [
    'monthly_revenue_trend.png',
    'quarterly_revenue_trend.png',
    'monthly_aov.png',
    'monthly_aov.svg',
    'top_10_countries_by_revenue.png',
    'top_10_countries_by_orders.png',
    'top_10_products_by_units.png',
    'top_10_products_by_revenue.png',
    'correlation_heatmap.png',
]

missing = [name for name in required_outputs if not (OUT / name).exists()]
if missing:
    raise FileNotFoundError(f'Missing required output files: {missing}')

print('All required visualisations exist in:', OUT)
for name in required_outputs:
    print('✓', name)